# Task 2: Contracts, nested data and a star schema
CSEN1095 Data Engineering · W26

In [ ]:
DATA = "bookings.jsonl"

In [5]:
import json, math, os, shutil
import datetime as dt
from typing import List, Literal

import numpy as np
import pandas as pd
import duckdb
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

## Part A: The contract

In [6]:
# extra="allow" so a new (drifting) field does not make a booking invalid: it is kept and reported later.
class Ticket(BaseModel):
    model_config = ConfigDict(extra="allow")
    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1, strict=True)
    price: float = Field(ge=0, strict=True)


class Student(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str = Field(pattern=r"^[0-9]{2}-[0-9]{4}$")
    name: str
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]


class Event(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str
    title: str
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: dt.date
    venue: str


class Booking(BaseModel):
    model_config = ConfigDict(extra="allow")
    booking_id: str = Field(min_length=1)
    booked_at: dt.datetime
    student: Student
    event: Event                                   # required: missing/None fails
    tickets: List[Ticket] = Field(min_length=1)    # at least one ticket
    total: float = Field(strict=True)
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]

    # cross-field rule 1: total == sum(qty * price)
    @model_validator(mode="after")
    def total_matches_tickets(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if not math.isclose(self.total, expected, abs_tol=0.01):
            raise ValueError("total does not equal the sum of qty x price")
        return self

    # cross-field rule 2: booked_at must be before the event's date
    @model_validator(mode="after")
    def booked_before_event(self):
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked_at is not before the event date")
        return self

In [7]:
def parse_line(line):
    """Return (raw_doc, Booking or None, reason or None). Reason = first error found."""
    if not line.strip():
        return None, None, "empty line"
    try:
        doc = json.loads(line)
    except json.JSONDecodeError:
        return None, None, "invalid JSON"
    if not isinstance(doc, dict):
        return None, None, "not a JSON object"
    try:
        return doc, Booking.model_validate(doc), None
    except ValidationError as e:
        err = e.errors()[0]
        loc = ".".join(str(p) for p in err["loc"] if not isinstance(p, int))  # drop list indices
        return doc, None, (f"{loc}: {err['msg']}" if loc else err["msg"])


with open(DATA, encoding="utf-8") as f:
    lines = f.read().splitlines()

raw_docs = {}      # line number -> raw dict (used for the schema-drift check)
valid = []         # (line number, raw dict, Booking)
quarantine = []    # {"line", "booking_id", "reason"}
seen_ids = set()

for line_no, line in enumerate(lines, start=1):
    doc, model, reason = parse_line(line)
    if doc is not None:
        raw_docs[line_no] = doc
    bid = doc.get("booking_id") if doc else None
    if model is None:
        quarantine.append({"line": line_no, "booking_id": bid, "reason": reason})
    elif model.booking_id in seen_ids:                       # re-sent booking: keep the first valid copy
        quarantine.append({"line": line_no, "booking_id": bid, "reason": "duplicate booking_id"})
    else:
        seen_ids.add(model.booking_id)
        valid.append((line_no, doc, model))

with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q) + "\n")

In [8]:
reason_counts = pd.Series([q["reason"] for q in quarantine], dtype="object").value_counts()
print(f"Lines read:  {len(lines)}")
print(f"Passed:      {len(valid)}")
print(f"Quarantined: {len(quarantine)}")
print("\nQuarantined per reason:")
print(reason_counts.to_string())
assert len(valid) + len(quarantine) == len(lines)

Lines read:  283
Passed:      259
Quarantined: 24

Quarantined per reason:
student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$'        4
Value error, total does not equal the sum of qty x price             4
tickets.qty: Input should be greater than or equal to 1              3
tickets.type: Input should be 'standard', 'vip' or 'student'         3
duplicate booking_id                                                 3
Value error, booked_at is not before the event date                  3
event: Input should be a valid dictionary or instance of Event       2
tickets: List should have at least 1 item after validation, not 0    2


## Part B: Flatten

In [9]:
core = [m.model_dump(mode="json") for _, _, m in valid]   # extras (drift fields) are included

# bookings: one row per booking, nested objects become columns, no tickets column
bookings = pd.json_normalize([{k: v for k, v in d.items() if k != "tickets"} for d in core], sep="_")
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True).dt.tz_localize(None)  # UTC, naive
bookings["event_date"] = pd.to_datetime(bookings["event_date"])

# booking_tickets: one row per ticket entry
booking_tickets = pd.json_normalize(core, record_path="tickets", meta=["booking_id"])
first = ["booking_id", "type", "qty", "price"]
booking_tickets = booking_tickets[first + [c for c in booking_tickets.columns if c not in first]]
booking_tickets.insert(4, "amount", booking_tickets["qty"] * booking_tickets["price"])

print("bookings:", bookings.shape, "| booking_tickets:", booking_tickets.shape)
bookings.head()

bookings: (259, 14) | booking_tickets: (314, 5)


,booking_id,booked_at,total,payment,status,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,promo_code
0,B0001,2026-09-28 15:46:00,340.0,card,cancelled,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,NaN
1,B0002,2026-09-27 12:29:00,40.0,card,confirmed,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,NaN
2,B0003,2026-10-13 20:19:00,60.0,wallet,confirmed,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,NaN
3,B0004,2026-11-01 06:42:00,105.0,card,confirmed,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage,CLUB10
4,B0005,2026-10-28 19:06:00,20.0,cash,confirmed,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A,None


In [10]:
# Checks
assert bookings["booking_id"].is_unique

per_booking = (bookings.set_index("booking_id")[["total"]]
               .join(booking_tickets.groupby("booking_id")["amount"].sum().rename("tickets_amount")))
assert per_booking["tickets_amount"].notna().all(), "a booking has no ticket rows"
assert np.isclose(per_booking["total"], per_booking["tickets_amount"], atol=0.01).all()
print("Both assertions passed")

Both assertions passed


In [11]:
# Schema drift: find every key path that is not in the example document in the task.
EXPECTED = {
    "booking_id", "booked_at", "total", "payment", "status",
    "student", "student.id", "student.name", "student.faculty",
    "event", "event.id", "event.title", "event.club", "event.date", "event.venue",
    "tickets", "tickets[].type", "tickets[].qty", "tickets[].price",
}

def key_paths(obj, prefix=""):
    out = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else k
            out.add(p)
            out |= key_paths(v, p)
    elif isinstance(obj, list):
        for item in obj:
            out |= key_paths(item, prefix + "[]")
    return out

rows = []
for line_no, doc in raw_docs.items():
    ts = doc.get("booked_at")
    ts = pd.to_datetime(ts, utc=True, errors="coerce") if isinstance(ts, str) else pd.NaT
    rows.append({"line": line_no, "booked_at": ts, "extra": key_paths(doc) - EXPECTED})

drift_fields = sorted(set().union(*[r["extra"] for r in rows]))
print("Unexpected fields:", drift_fields or "none")

for field in drift_fields:
    with_f = [r for r in rows if field in r["extra"]]
    without_f = [r for r in rows if field not in r["extra"]]
    ts_with = [r["booked_at"] for r in with_f if pd.notna(r["booked_at"])]
    ts_without = [r["booked_at"] for r in without_f if pd.notna(r["booked_at"])]
    print(f"\n'{field}': in {len(with_f)} documents, missing in {len(without_f)}")
    print("  earliest booked_at WITH the field:   ", min(ts_with) if ts_with else None)
    print("  latest booked_at WITHOUT the field:  ", max(ts_without) if ts_without else None)
    print("  first line number with the field:    ", min(r["line"] for r in with_f))
    if ts_with and ts_without:
        print("  clean cut-over by date?              ", max(ts_without) < min(ts_with))

# What the code does with documents that lack the field: json_normalize gives them a null.
for field in drift_fields:
    col = field.replace(".", "_")
    if col in bookings.columns:
        print(f"bookings['{col}']: {bookings[col].notna().sum()} filled, {bookings[col].isna().sum()} null")

Unexpected fields: ['promo_code']

'promo_code': in 193 documents, missing in 90
  earliest booked_at WITH the field:    2026-10-20 01:05:00+00:00
  latest booked_at WITHOUT the field:   2026-10-31 10:00:00+00:00
  first line number with the field:     4
  clean cut-over by date?               False
bookings['promo_code']: 87 filled, 172 null


## Part C: Model and store

In [12]:
# Is each event / student described identically in every booking? (the dimension keeps the earliest booking's values)
ev_cols = ["event_id", "event_title", "event_club", "event_date", "event_venue"]
st_cols = ["student_id", "student_name", "student_faculty"]
print("events with conflicting attributes:  ", bookings[ev_cols].drop_duplicates()["event_id"].duplicated().sum())
print("students with conflicting attributes:", bookings[st_cols].drop_duplicates()["student_id"].duplicated().sum())

con = duckdb.connect()
con.register("bookings", bookings)
con.register("booking_tickets", booking_tickets)

con.execute("""
CREATE TABLE dim_event AS
SELECT row_number() OVER (ORDER BY event_id) AS event_key,
       event_id, event_title, event_club, event_date, event_venue
FROM bookings
QUALIFY row_number() OVER (PARTITION BY event_id ORDER BY booked_at, booking_id) = 1
""")

con.execute("""
CREATE TABLE dim_student AS
SELECT row_number() OVER (ORDER BY student_id) AS student_key,
       student_id, student_name, student_faculty
FROM bookings
QUALIFY row_number() OVER (PARTITION BY student_id ORDER BY booked_at, booking_id) = 1
""")

con.execute("""
CREATE TABLE fact_ticket_sales AS
SELECT t.booking_id,
       e.event_key,
       s.student_key,
       t."type"  AS ticket_type,
       t.qty,
       t.price,
       t.amount,
       b.status,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS date_key,
       strftime(b.booked_at, '%Y-%m')                   AS booked_month
FROM booking_tickets t
JOIN bookings b      ON b.booking_id = t.booking_id
JOIN dim_event e     ON e.event_id   = b.event_id
JOIN dim_student s   ON s.student_id = b.student_id
""")

for t in ["dim_event", "dim_student", "fact_ticket_sales"]:
    print(t, con.execute(f"SELECT count(*) FROM {t}").fetchone()[0], "rows")

events with conflicting attributes:   0
students with conflicting attributes: 0
dim_event 8 rows
dim_student 103 rows
fact_ticket_sales 314 rows


In [13]:
# Checks
n_fact = con.execute("SELECT count(*) FROM fact_ticket_sales").fetchone()[0]
assert n_fact == len(booking_tickets)

fact_amount = con.execute("SELECT sum(amount) FROM fact_ticket_sales").fetchone()[0]
assert math.isclose(fact_amount, bookings["total"].sum(), abs_tol=0.01)
print(f"fact rows = {n_fact} = booking_tickets rows;  SUM(amount) = {fact_amount:.2f} = SUM(total) in bookings")

fact rows = 314 = booking_tickets rows;  SUM(amount) = 31285.00 = SUM(total) in bookings


In [14]:
# Confirmed bookings only: tickets sold and revenue per club and ticket type
answer = con.execute("""
SELECT e.event_club AS club,
       f.ticket_type,
       sum(f.qty)    AS tickets_sold,
       sum(f.amount) AS revenue
FROM fact_ticket_sales f
JOIN dim_event e ON e.event_key = f.event_key
WHERE f.status = 'confirmed'
GROUP BY e.event_club, f.ticket_type
ORDER BY e.event_club, f.ticket_type
""").df()
answer

,club,ticket_type,tickets_sold,revenue
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


In [15]:
# Save as Parquet: one file per dimension, the fact partitioned by booking month
shutil.rmtree("star", ignore_errors=True)
os.makedirs("star")
con.execute("COPY dim_event   TO 'star/dim_event.parquet'   (FORMAT PARQUET)")
con.execute("COPY dim_student TO 'star/dim_student.parquet' (FORMAT PARQUET)")
con.execute("COPY fact_ticket_sales TO 'star/fact_ticket_sales' (FORMAT PARQUET, PARTITION_BY (booked_month))")

for root, _, files in sorted(os.walk("star")):
    for fn in sorted(files):
        print(os.path.join(root, fn))

back = con.execute("SELECT count(*) FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning = true)").fetchone()[0]
assert back == n_fact
print("Read back from Parquet:", back, "fact rows")

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet
Read back from Parquet: 314 fact rows


## Part D: Numbers for the README paragraph

In [17]:
# What if we removed duplicate booking_ids BEFORE validating? (keep the first line per booking_id, valid or not)
seen, valid_swapped, dup_swapped = set(), 0, 0
for line in lines:
    doc, model, reason = parse_line(line)
    bid = doc.get("booking_id") if doc else None
    if isinstance(bid, str):
        if bid in seen:
            dup_swapped += 1
            continue
        seen.add(bid)
    if model is not None:
        valid_swapped += 1

dup_now = int((pd.Series([q["reason"] for q in quarantine], dtype="object") == "duplicate booking_id").sum())
print(f"Validate first, then de-duplicate: {len(valid)} bookings left ({dup_now} duplicates)")
print(f"De-duplicate first, then validate: {valid_swapped} bookings left ({dup_swapped} duplicates)")
print("Counts change if swapped:", valid_swapped != len(valid))

# What would SUM(total) over the fact give? Each booking's total is repeated on every ticket row of that booking.
wrong = con.execute("""
SELECT sum(b.total) FROM fact_ticket_sales f JOIN bookings b ON b.booking_id = f.booking_id
""").fetchone()[0]
print(f"\nSUM(total) over the fact = {wrong:.2f}  vs  true sum of totals = {bookings['total'].sum():.2f}")
print("Bookings with more than one ticket row:", int((booking_tickets.groupby('booking_id').size() > 1).sum()))

Validate first, then de-duplicate: 259 bookings left (3 duplicates)
De-duplicate first, then validate: 259 bookings left (3 duplicates)
Counts change if swapped: False

SUM(total) over the fact = 42000.00  vs  true sum of totals = 31285.00
Bookings with more than one ticket row: 55
